# Интро

In [35]:
import math
import shap
import optuna
import pandas as pd
import numpy as np
import category_encoders as ce
import seaborn as sns
from sklearn.svm import SVC
import matplotlib.pyplot as plt
from scipy.stats import shapiro
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
# from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import cross_val_score, StratifiedKFold
from imblearn.over_sampling import SMOTE, ADASYN, BorderlineSMOTE, SVMSMOTE, RandomOverSampler
from sklearn.metrics import roc_curve, recall_score, confusion_matrix, precision_score, f1_score, accuracy_score, classification_report, roc_auc_score

In [36]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer, make_column_selector

In [37]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from src.preprocess_funsc import drop_nan_cols, feature_engineering

In [38]:
df = pd.read_csv(r'../datasets/train.csv')

# предобработка данных 

удаление колонки с индексами

In [39]:
df = pd.read_csv(r'../datasets/train.csv')
y = df['TARGET']
df = df.drop(columns=['TARGET', 'ID'])

обработка пропусков

In [40]:
def drop_nan_cols(df, alpha=0.5):
    '''
    дропает колонки в которых доля пропущенных значений > alpha
    '''
    alpha_count = df.shape[0] * alpha
    cols = df.isna().sum()[df.isna().sum() > alpha_count].index
    cols = list(cols)
    dfc = df.drop(columns=cols)
    return dfc

In [41]:
a = drop_nan_cols(df)
n_cols = a.shape[1]
# a.isna().sum()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(a, y, stratify=y, test_size=0.2, random_state=1)

# feature engineering

In [43]:
X_train = feature_engineering(X_train)
X_test = feature_engineering(X_test)
X_train.head()

,CR_PROD_CNT_IL,AMOUNT_RUB_CLO_PRC,TURNOVER_DYNAMIC_IL_1M,AMOUNT_RUB_SUP_PRC,REST_DYNAMIC_FDEP_1M,REST_DYNAMIC_SAVE_3M,CR_PROD_CNT_VCU,REST_AVG_CUR,AMOUNT_RUB_NAS_PRC,CLNT_JOB_POSITION,...,CLNT_SETUP_TENOR,TURNOVER_DYNAMIC_PAYM_3M,TURNOVER_DYNAMIC_PAYM_1M,TRANS_AMOUNT_TENDENCY3M,TRANS_CNT_TENDENCY3M,REST_DYNAMIC_CC_1M,TURNOVER_DYNAMIC_CC_1M,REST_DYNAMIC_CC_3M,TURNOVER_DYNAMIC_CC_3M,total_products
162608,0,0.270151,0.0,0.007900,0.0,0.0,0,89264.527667,0.000000,главный кредитный специалист,...,1.149467,0.000000,0.00000,0.687720,0.621212,0.0,0.0,0.0,0.0,0
202572,0,0.003185,0.0,0.434819,0.0,0.0,0,107135.377867,0.137162,Индивидуальный предприниматель,...,1.496241,0.000000,0.00000,0.566899,0.525140,0.0,0.0,0.0,0.0,0
119711,0,0.024661,0.0,0.005203,0.0,0.0,0,0.000000,0.000000,NaN,...,4.969359,0.633592,0.19289,0.598663,0.583333,0.0,0.0,0.0,0.0,0
81302,0,0.000000,0.0,0.000000,0.0,0.0,0,18586.184067,0.000000,МЕНЕДЖЕР,...,1.144098,0.000000,0.00000,NaN,NaN,0.0,0.0,0.0,0.0,0
83142,2,0.000000,0.0,0.000000,0.0,0.0,1,7425.564500,0.000000,ПОВАР,...,4.050012,0.000000,0.00000,1.000000,1.000000,0.0,0.0,0.0,0.0,4


кодирование категориальных признаков
и маштабирование числовых

In [44]:
cat_cols = X_train.select_dtypes(['string']).columns.to_list()
fl_cols = X_train.select_dtypes(['float']).columns.to_list()
int_cols = X_train.select_dtypes(['int']).columns.to_list()

In [45]:
cat_pipline = Pipeline([
    ('cat_imputer', SimpleImputer(strategy='most_frequent')),
    ('cat_encoder', ce.TargetEncoder()),
])

num_pipline = Pipeline([
    ('num_imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler()),
])

prep = ColumnTransformer(
    transformers=[
        ('cat', cat_pipline, make_column_selector(dtype_include=['str'])),
        ('num', num_pipline, make_column_selector(dtype_include=['number']))
    ],
    verbose_feature_names_out=False,
).set_output(transform='pandas')

X_train_1 = prep.fit_transform(X_train, y_train)
X_test_1 = prep.transform(X_test)


проверка на дубликаты  

In [46]:
print('количество дублирующихся строк в трейне', X_train_1.duplicated().sum())
train_dupl_idx = X_train_1[X_train_1.duplicated()].index
X_train_1 = X_train_1.drop_duplicates()
y_train = y_train.drop(index=train_dupl_idx)


количество дублирующихся строк в трейне 77


# ML predictions

baseline на датасете

In [51]:
lr = LogisticRegression(random_state=1)
lr.fit(X_train_1, y_train)
y_pred = lr.predict(X_test_1)
y_proba = lr.predict_proba(X_test_1)
accuracy_sc = accuracy_score(y_test, y_pred) 
recall_sc = recall_score(y_test, y_pred)
rocauc = roc_auc_score(y_test, y_proba[:, 1])

print('Accureacy:', accuracy_sc)
print('Recall:', recall_sc)
print('ROC AUC:', rocauc)

Accureacy: 0.916537627748529
Recall: 0.01676750216076059
ROC AUC: 0.7166923091625508


Main

In [52]:
rf = RandomForestClassifier(n_estimators=100, random_state=1)
rf.fit(X_train_1, y_train)
y_pred = rf.predict(X_test_1)
y_proba = rf.predict_proba(X_test_1)
accuracy_sc = accuracy_score(y_test, y_pred) 
recall_sc = recall_score(y_test, y_pred)
rocauc = roc_auc_score(y_test, y_proba[:, 1])

print('Accureacy:', accuracy_sc)
print('Recall:', recall_sc)
print('ROC AUC:', rocauc)

Accureacy: 0.9163546271009883
Recall: 0.03612791702679343
ROC AUC: 0.8213737683552065


In [56]:
lgbm = LGBMClassifier(n_estimators=100, random_state=1, verbose=-1)
lgbm.fit(X_train_1, y_train)
y_pred = lgbm.predict(X_test_1)
y_proba = lgbm.predict_proba(X_test_1)
accuracy_sc = accuracy_score(y_test, y_pred) 
recall_sc = recall_score(y_test, y_pred)
rocauc = roc_auc_score(y_test, y_proba[:, 1])

print('Accureacy:', accuracy_sc)
print('Recall:', recall_sc)
print('ROC AUC:', rocauc)

Accureacy: 0.917100706664039
Recall: 0.04477095937770095
ROC AUC: 0.827212767124454


In [53]:
cat_bst = CatBoostClassifier(iterations=100, random_seed=1, verbose=0)
cat_bst.fit(X_train_1, y_train)
y_pred = cat_bst.predict(X_test_1)
y_proba = cat_bst.predict_proba(X_test_1)
accuracy_sc = accuracy_score(y_test, y_pred) 
recall_sc = recall_score(y_test, y_pred)
rocauc = roc_auc_score(y_test, y_proba[:, 1])

print('Accureacy:', accuracy_sc)
print('Recall:', recall_sc)
print('ROC AUC:', rocauc)

Accureacy: 0.9157633942397027
Recall: 0.07657735522904062
ROC AUC: 0.8179208800223255


## Лучшая модель

выбираю лучшую модель и подбираю гиперпараметры

In [57]:
def objective(trial):
    lgmb_pars = {
        'n_estimators':  trial.suggest_int('n_estimators', 10, 1000),
        'num_leaves': trial.suggest_int('num_leaves', 10, 100),
        'learning_rate': trial.suggest_float('learning_rate', 1e-5, 0.1),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 1000)
    }
    lgbm = LGBMClassifier(**lgmb_pars)
    lgbm.fit(X_train_1, y_train)
    y_proba = lgbm.predict_proba(X_test_1)
    roc_auc = roc_auc_score(y_test, y_proba[:, 1])
    return roc_auc

In [59]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30, show_progress_bar=True)
print(study.best_params)
print(study.best_value)


[I 2026-10-07 22:22:37,575] A new study created in memory with name: no-name-4511ba0c-13ef-4184-9474-6fe81e499bc2


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-10-07 22:22:40,299] Trial 0 finished with value: 0.8260348415020369 and parameters: {'n_estimators': 103, 'num_leaves': 85, 'learning_rate': 0.048872509211302845, 'min_child_samples': 432}. Best is trial 0 with value: 0.8260348415020369.
[I 2026-10-07 22:22:49,892] Trial 1 finished with value: 0.8241706978148386 and parameters: {'n_estimators': 789, 'num_leaves': 57, 'learning_rate': 0.08492626727887205, 'min_child_samples': 365}. Best is trial 0 with value: 0.8260348415020369.
[I 2026-10-07 22:23:01,766] Trial 2 finished with value: 0.8239692228590584 and parameters: {'n_estimators': 672, 'num_leaves': 87, 'learning_rate': 0.08541775297047748, 'min_child_samples': 781}. Best is trial 0 with value: 0.8260348415020369.
[I 2026-10-07 22:23:08,944] Trial 3 finished with value: 0.8267380931935681 and parameters: {'n_estimators': 470, 'num_leaves': 40, 'learning_rate': 0.018321479993685268, 'min_child_samples': 740}. Best is trial 3 with value: 0.8267380931935681.
[I 2026-10-07 22:2